<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/InceptionV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Working on: {device}")

Working on: cuda


In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from sklearn.utils.class_weight import compute_class_weight

In [6]:
# Load the processed .npy files
X_train = np.load('X_train.npy')
Y_train_lab = np.load('Y_train_labels.npy')
X_test = np.load('X_test.npy')
Y_test_lab = np.load('Y_test_labels.npy')

In [14]:
print(X_train.shape)
print(Y_train_lab.shape)
print(X_test.shape)
print(Y_test_lab.shape)

(364, 224, 224, 6)
(364,)
(92, 224, 224, 6)
(92,)


In [15]:
# Map Labels to 0-indexed
unique_labels = np.unique(Y_train_lab)
label_map = {old_label: new_label for new_label, old_label in enumerate(unique_labels)}
Y_train_ready = np.array([label_map[l] for l in Y_train_lab])
Y_test_ready = np.array([label_map[l] for l in Y_test_lab])

In [16]:
print("--- LABEL MAPPING KEY ---")

class_map = {
    10: ("Tree cover", "#006400"),
    20: ("Shrubland", "#ffbb22"),
    30: ("Grassland", "#ffff4c"),
    40: ("Cropland", "#f096ff"),
    50: ("Built-up", "#fa0000"),
    60: ("Bare / Sparse vegetation", "#b4b4b4"),
    70: ("Snow and ice", "#f0f0f0"),
    80: ("Permanent water bodies", "#0064ff"),
    90: ("Herbaceous wetland", "#0096a0"),
}

# Sorting by the new index (0, 1, 2...) for readability
for old_id, new_id in sorted(label_map.items(), key=lambda item: item[1]):
    # Get the name from our previous class_map, default to "Unknown" if not found
    class_name = class_map.get(old_id, ("Unknown", ""))[0]
    print(f"New ID: {new_id}  <--  Original ID: {old_id} ({class_name})")

# Also, let's verify the shapes to be 100% sure before training
print("\n--- DATA SHAPE VERIFICATION ---")
print(f"X_train: {X_train.shape}")
print(f"Y_train_ready: {Y_train_ready.shape}")
print(f"Unique classes in Training: {np.unique(Y_train_ready)}")

--- LABEL MAPPING KEY ---
New ID: 0  <--  Original ID: 10 (Tree cover)
New ID: 1  <--  Original ID: 20 (Shrubland)
New ID: 2  <--  Original ID: 30 (Grassland)
New ID: 3  <--  Original ID: 40 (Cropland)
New ID: 4  <--  Original ID: 50 (Built-up)
New ID: 5  <--  Original ID: 80 (Permanent water bodies)

--- DATA SHAPE VERIFICATION ---
X_train: (364, 224, 224, 6)
Y_train_ready: (364,)
Unique classes in Training: [0 1 2 3 4 5]


In [17]:
# 1. Define the 6-band Input (B2, B3, B4, B8, B11, B12)
input_6band = Input(shape=(224, 224, 6))

# 2. The 1x1 Convolution (The "Adapter")
# This layer learns the spectral relationship between your 6 bands
x = Conv2D(3, (1, 1), padding='same', name='band_adapter')(input_6band)

# 3. Load InceptionV3 base with 3-channel input shape
# Note: we don't pass the input_tensor here to avoid the layer mismatch error
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 4. Connect the adapter output to the base model
x = base_model(x)

# 5. Add the Classification Head
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(len(unique_labels), activation='softmax')(x)

# 6. Final Model Construction
model = Model(inputs=input_6band, outputs=predictions)

print("Model architecture defined with 6-band adapter.")

Model architecture defined with 6-band adapter.


In [20]:
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import f1_score
import numpy as np

class MetricsTable(Callback):
    def __init__(self, x_val, y_val):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val

    def on_train_begin(self, logs=None):
        print(f"\n{'Epoch':<6} | {'Train Loss':<12} | {'Val Loss':<10} | {'Acc':<8} | {'F1 (Macro)':<10}")
        print("-" * 65)

    def on_epoch_end(self, epoch, logs=None):
        # 1. Use the data we passed in during initialization
        val_pred_probs = self.model.predict(self.x_val, verbose=0)
        val_pred = np.argmax(val_pred_probs, axis=1)
        val_true = self.y_val

        # 2. Calculate Macro F1
        f1 = f1_score(val_true, val_pred, average='macro')

        # 3. Pull metrics from logs
        train_loss = logs.get('loss', 0)
        val_loss = logs.get('val_loss', 0)
        val_acc = logs.get('val_accuracy', 0)

        print(f"{epoch+1:<6} | {train_loss:<12.4f} | {val_loss:<10.4f} | {val_acc:<8.4f} | {f1:<10.4f}")

# IMPORTANT: Re-initialize the callback with your test data
metrics_callback = MetricsTable(X_test, Y_test_ready)

In [21]:
# Freeze the InceptionV3 weights
base_model.trainable = False

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Calculate weights to balance the 6 major classes
weights = compute_class_weight('balanced', classes=np.unique(Y_train_ready), y=Y_train_ready)
class_weight_dict = dict(enumerate(weights))


print("Starting Phase 1: Training Adapter only")
history_warmup = model.fit(
    X_train, Y_train_ready,
    epochs=5,
    batch_size=16,
    validation_data=(X_test, Y_test_ready),
    class_weight=class_weight_dict,
    callbacks=[metrics_callback],
    verbose=0
)

Starting Phase 1: Training Adapter only

Epoch  | Train Loss   | Val Loss   | Acc      | F1 (Macro)
-----------------------------------------------------------------
1      | 1.4734       | 0.9906     | 0.6304   | 0.4214    
2      | 0.9987       | 0.5357     | 0.8370   | 0.6646    
3      | 0.7807       | 0.6146     | 0.7717   | 0.6418    
4      | 0.5339       | 0.4175     | 0.8804   | 0.7075    
5      | 0.4929       | 0.4142     | 0.8696   | 0.7159    


In [22]:
# Unfreeze the whole model
base_model.trainable = True

# Use a very low learning rate to avoid destroying pre-trained patterns
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n Starting Phase 2: Fine-tuning entire model")
history_finetune = model.fit(
    X_train, Y_train_ready,
    epochs=10,
    batch_size=16,
    validation_data=(X_test, Y_test_ready),
    class_weight=class_weight_dict,
    callbacks=[metrics_callback],
    verbose=0
)


 Starting Phase 2: Fine-tuning entire model

Epoch  | Train Loss   | Val Loss   | Acc      | F1 (Macro)
-----------------------------------------------------------------
1      | 2.3913       | 0.4581     | 0.8478   | 0.6770    
2      | 1.7518       | 0.5083     | 0.8370   | 0.6563    
3      | 1.4336       | 0.5506     | 0.8043   | 0.4794    
4      | 1.2164       | 0.5365     | 0.8261   | 0.4891    
5      | 1.0535       | 0.5388     | 0.8370   | 0.5613    
6      | 0.8151       | 0.5545     | 0.8478   | 0.5667    
7      | 0.8877       | 0.5754     | 0.8043   | 0.5044    
8      | 0.7365       | 0.5708     | 0.7935   | 0.5303    
9      | 0.6082       | 0.5712     | 0.8152   | 0.5511    
10     | 0.6537       | 0.5688     | 0.7935   | 0.4758    


In [23]:
from sklearn.metrics import classification_report, accuracy_score

train_preds = np.argmax(model.predict(X_train), axis=1)
test_preds = np.argmax(model.predict(X_test), axis=1)

# Print Overall Accuracies
print(f"Train Accuracy: {accuracy_score(Y_train_ready, train_preds):.4f}")
print(f"Test Accuracy:  {accuracy_score(Y_test_ready, test_preds):.4f}")

# Detailed Classification Report
target_names = [class_map[old_id][0] for old_id, new_id in sorted(label_map.items(), key=lambda x: x[1])]

print("\nClassification Report:\n")
print(classification_report(Y_test_ready, test_preds, target_names=target_names))

12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 392ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Train Accuracy: 0.8874
Test Accuracy:  0.7935

Classification Report:

                        precision    recall  f1-score   support

            Tree cover       0.50      0.70      0.58        10
             Shrubland       0.33      0.60      0.43         5
             Grassland       0.00      0.00      0.00         5
              Cropland       0.00      0.00      0.00         1
              Built-up       0.94      0.87      0.91        55
Permanent water bodies       0.94      0.94      0.94        16

              accuracy                           0.79        92
             macro avg       0.45      0.52      0.48        92
          weighted avg       0.80      0.79      0.79        92

